# 04 · Supplemental sources

Builds on notebook 03's carry-forward set (baseline + spatial context + owner track
record + nearby outcomes; DCAD building systems dropped). Each source is added on its
own, then in combination. Same splits, models and metrics (`holdout.py`).

Every feature carries a **timing class** relative to the FY2020-21 to FY2024-25
inspection window. Only *pre* features are clean under the Chelsea premise; *during* and
*snapshot* results are reported separately and should be read as upper bounds.

| Source | Features | Timing | Join | Docs |
|---|---|---|---|---|
| Eviction filings resolved to parcels (`dev_sandbox.taylor.evictions_north_tx_landlord`) | filings per 100 units per year on the property's parcel; strong/good tier only; plus adjacent parcels (≤ 600 ft) with the same owner of record | pre (2017–2020) and during (2021–2025) | DCAD `gis_parcel_id` | `housing-docs/methods/property-owner-resolution.md`, `filing-to-parcel-resolution.md` |
| DCAD certified roll history (`dev_silver.silver_tx_dallas_cad_car.account`, `deed`, `account_owner`) | market value growth 2016→2021 and 2019→2021, share of years with a tax-protest agent, rendition penalty, years since last sale, sales 2016–2020, distinct owner names 2016–2021; sold during the window | pre; `sold_during_window` is during | `universal_id` | cpal-data-platform README (DCAD) |
| DCTO tax roll (`silver.silver_tx_dallas_tax.tax_account_summary`) | still delinquent from 2020 or earlier; delinquent, delinquent years and tax suit in 2026 | pre (oldest unpaid ≤ 2020) and snapshot | `universal_id` | `housing-docs` DCTO tax roll data dictionary |
| CoStar multifamily (`gold.housing.costar_03312026`) | building class, star rating, affordable flag, average unit size, years since renovation; CoStar true owner as a fourth owner grouping; rent per unit and vacancy | snapshot (2026-03); class and stars move slowly, rent and vacancy do not | DCAD account, else nearest CoStar point ≤ 75 m | `cad-data/docs/source_runbooks/costar.md` |
| DPD NIBRS incidents at apartment premises (`gold.safety.nibrs_crime`) | incidents within 150 m per 100 units per year; violent subset | during (2023-01 onward only) | coordinates | none on the table |
| Dallas 311, Code Compliance (Dallas Open Data `4n3h-rfqm`, pulled by `pull_311.py`) | resident-initiated requests within 100 m per 100 units per year; Emergency-priority subset; hazard-type subset (A/C or heat inoperable, life hazard, substandard structure, rodent, open and vacant), which exists only through 2019 because from 2020 the city files nearly everything as "Code Concern - CCS" | pre (2017-01 to 2020-09) and during | coordinates | dataset page on Dallas Open Data |

CoStar values are licensed and stay internal; only aggregate metrics leave this notebook.
The HFC/PFC roster (`dev_sandbox.taylor.hfc_pfc_roster`) was checked and dropped: its
account and parcel keys are empty for 233 of 249 rows and none match an inspected property.

**Leakage guards for 311.** Graded inspections are themselves service requests. Requests
whose number matches an inspection are removed, and only resident-initiated requests
count (method Proactive and Internal are dropped).

## Findings

Run 2026-10-01. Outputs are cleared before commit; the numbers below are from that run.

Ensemble PR-AUC for `bottom10`. Test prevalence is 0.163 on the Chelsea mirror, 0.197 on the
random split and 0.196 on grouped CV.

| Configuration | Timing | Chelsea mirror | Random 70/30 | Grouped CV by tract |
|---|---|---|---|---|
| 03 carry-forward | pre | 0.308 | 0.438 | 0.328 |
| + evictions | pre | 0.313 | 0.444 | 0.344 |
| + DCAD history | pre | 0.336 | 0.448 | 0.345 |
| + 311 | pre | 0.321 | 0.436 | 0.329 |
| + CoStar class/stars + owner | snapshot | 0.301 | 0.435 | 0.331 |
| **all pre-window** | pre | **0.331** | **0.467** | **0.358** |
| all pre-window + CoStar | snapshot | 0.332 | 0.450 | 0.358 |
| + evictions, during | during | 0.346 | 0.447 | 0.367 |
| + 311, during | during | 0.333 | 0.450 | 0.364 |
| + crime, during | during | 0.331 | 0.452 | 0.362 |
| everything | mixed | 0.344 | 0.461 | 0.376 |

- **The clean (pre-window) model** reaches 0.33 on the Chelsea mirror, about twice the 0.16
  base rate. Inspecting its top 20% finds 42% of the test area's bottom-10 properties
  (baseline 40%, random 20%). On grouped CV it reaches 0.36 against a 0.20 base rate.
- **DCAD ownership history** is the most useful new source: years since the last sale,
  sales in 2016–2020, distinct owner names and value growth. Bottom-10 properties had sold
  more recently (7.6 against 9.4 years before the window) and more often.
- **Evictions** carry signal (filings per unit run about 40% higher at bottom-10
  properties). Most of it is already captured by the other features. Concurrent filings
  (2021–2025) add 1–2 points over pre-window ones. On its own, ranking by concurrent filings
  per unit scores 0.26 on the mirror, nearly the whole 02 baseline model.
- **Resident 311 complaints** are weak. Ranked alone, concurrent resident complaints score
  0.20 on the mirror, barely above poverty alone (0.19) and random (0.16). Chelsea's
  argument was that a model beats reactive response; on that narrow comparison it does here
  too, by a wide margin. The specific hazard request types exist only through 2019. From
  2020 the city files nearly everything as "Code Concern - CCS".
- **CoStar** class, star rating and true owner add nothing over DCAD. Its rent and vacancy are
  2026 snapshots and enter only the "everything" row.
- **Apartment-premise crime** (2023 onward) adds nothing on top.
- **Tax delinquency** is too rare to matter. Only 0.7% of non-bottom-10 properties, and no
  bottom-10 property, were still delinquent from 2020 or earlier.
- **Everything, including concurrent and snapshot data,** reaches 0.34 on the mirror and 0.38
  on grouped CV. Read that as an upper bound for the spatial design.
- **Leak check:** shuffled labels score at prevalence (0.18–0.20).
- **The HFC/PFC roster was dropped.** Its keys are empty for 233 of 249 rows, and no row matches an
  inspected property.

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

import features as f
import holdout as h

warnings.filterwarnings("ignore", message="Skipping features without any observed values")
pd.set_option("display.float_format", "{:.3f}".format)

df = h.load_base()
CONTEXT = f.spatial_context(df)
EV = f.evictions(df)
HIST = f.dcad_history(df)
TAX = f.tax_delinquency(df)
COSTAR, costar_owner = f.costar(df)
CRIME = f.apartment_crime(df)
C311 = f.complaints_311(df)
df = pd.concat(
    [df, CONTEXT, f.owner_keys(df)[["log_sponsor_portfolio"]], EV, HIST, TAX, COSTAR, CRIME, C311],
    axis=1,
).assign(costar_owner=costar_owner)
print("rows:", len(df))

In [ ]:
new = pd.concat([EV, HIST, TAX, COSTAR, CRIME, C311], axis=1)
summary = pd.DataFrame({
    "coverage": new.notna().mean(),
    "mean | bottom10": new[df.bottom10 == 1].mean(),
    "mean | not": new[df.bottom10 == 0].mean(),
})
print(summary.round(3).to_string())
print("CoStar true owner on", costar_owner.notna().sum(), "properties,", costar_owner.nunique(), "distinct")

## Ablation

In [ ]:
C03 = h.BASELINE_FEATURES + list(CONTEXT.columns) + ["log_sponsor_portfolio"]
FOLD = f.combine(f.owner_track_record, f.nearby_outcomes)

EV_PRE = ["ev_pre", "ev_pre_strong", "ev_pre_adj"]
EV_DURING = ["ev_during", "ev_during_strong", "ev_during_adj"]
HIST_PRE = [c for c in HIST.columns if c != "sold_during_window"]
TAX_PRE, TAX_SNAP = ["delinquent_since_pre2021"], ["delinquent_2026", "n_delinquent_years_2026", "tax_suit_2026"]
COSTAR_SLOW = ["costar_class", "costar_stars", "costar_affordable", "costar_unit_sf", "costar_years_since_renovated"]
COSTAR_SNAP = ["costar_log_rent_per_unit", "costar_vacancy_2026"]
C311_PRE, C311_DURING = ["c311_pre", "c311_hazard_pre"], ["c311_during", "c311_emergency_during"]
CRIME_DURING = list(CRIME.columns)

no_costar_owner = df.drop(columns="costar_owner")
PRE_ALL = C03 + EV_PRE + HIST_PRE + TAX_PRE + C311_PRE
CONFIGS = {
    "03 carry-forward": (C03, no_costar_owner),
    "+ evictions, pre": (C03 + EV_PRE, no_costar_owner),
    "+ DCAD history, pre": (C03 + HIST_PRE, no_costar_owner),
    "+ 311, pre": (C03 + C311_PRE, no_costar_owner),
    "+ CoStar class/stars + owner": (C03 + COSTAR_SLOW, df),
    "all pre-window": (PRE_ALL, no_costar_owner),
    "all pre-window + CoStar": (PRE_ALL + COSTAR_SLOW, df),
    "+ evictions, during": (PRE_ALL + COSTAR_SLOW + EV_DURING, df),
    "+ 311, during": (PRE_ALL + COSTAR_SLOW + C311_DURING, df),
    "+ crime, during": (PRE_ALL + COSTAR_SLOW + CRIME_DURING, df),
    "everything": (
        PRE_ALL + COSTAR_SLOW + COSTAR_SNAP + EV_DURING + C311_DURING + CRIME_DURING
        + TAX_SNAP + ["sold_during_window"],
        df,
    ),
}
BASELINES = {"Reactive: resident 311, during": "c311_during", "Evictions, during": "ev_during_adj"}
results = pd.concat(
    [
        h.run(data, t, feats, fold_features=FOLD, baselines=BASELINES, label=name)
        for t in h.TARGETS
        for name, (feats, data) in CONFIGS.items()
    ],
    ignore_index=True,
)

In [ ]:
for t in h.TARGETS:
    r = results[results.target == t]
    print(f"\n=== {t}: PR-AUC (test prevalence: "
          + ", ".join(f"{s} {r[r.split == s].prevalence_test.iloc[0]:.3f}" for s in h.SPLITS) + ")")
    print(h.compare(r[r.ranker.isin(["Random forest", "XGBoost", "LASSO", "Ensemble"])]).round(3).to_string())

In [ ]:
for t in h.TARGETS:
    r = results[(results.target == t) & (results.label == "03 carry-forward")]
    print(f"\n=== {t}: single-column rankers")
    print(h.compare(r[~r.ranker.isin(["Random forest", "XGBoost", "LASSO", "Ensemble"])]).round(3).to_string())

In [ ]:
for t in h.TARGETS:
    print(f"\n=== {t}: share of bottom-10 found in the top 20% (Ensemble)")
    r = results[(results.target == t) & (results.ranker == "Ensemble")]
    print(r.pivot_table(index="label", columns="split", values="found@20%", sort=False)[h.SPLITS].round(3).to_string())

## Leak check

In [ ]:
rng = np.random.default_rng(h.SEED)
feats, data = CONFIGS["everything"]
shuffled = data.assign(bottom10=rng.permutation(data.bottom10.to_numpy()))
chk = h.run(shuffled, "bottom10", feats, fold_features=FOLD,
            which={"Chelsea mirror", "Random 70/30"}, label="shuffled")
print(chk[["split", "ranker", "prevalence_test", "PR-AUC"]].to_string(index=False))

## What carries the model (Chelsea mirror, `bottom10`)

In [ ]:
for name in ["all pre-window + CoStar", "everything"]:
    feats, data = CONFIGS[name]
    est, X_te, y_te = h.fit_split(data, "bottom10", feats, "Chelsea mirror", "Random forest", fold_features=FOLD)
    r = permutation_importance(est, X_te, y_te, scoring="average_precision", n_repeats=20,
                               random_state=h.SEED, n_jobs=h.N_JOBS)
    imp = pd.Series(r.importances_mean, index=X_te.columns).sort_values(ascending=False)
    print(f"\n=== {name}")
    print(imp.head(15).round(4).to_string())